In [1]:
import pandas as pd
import numpy as np

1
List the total number of unique movies, genres, cast members and crew members collected across all tables.

In [12]:
movies_df = pd.read_csv("movies_cleaned.csv")
genres_df = pd.read_csv("genres_cleaned.csv")
cast_df = pd.read_csv("cast_cleaned.csv")
crew_df = pd.read_csv("crew_cleaned.csv")

print("Total Movies:", movies_df["movie_id"].nunique())
print("Total Genres:", genres_df["genre_id"].nunique())
print("Total Cast Members:", cast_df["person_id"].nunique())
print("Total Crew Members:", crew_df["person_id"].nunique())

Total Movies: 2503
Total Genres: 19
Total Cast Members: 11300
Total Crew Members: 2848


2
List all movies with missing or zero budget/revenue values, and decide how these should be treated for the rest of the analysis.



In [13]:
q2 = movies_df[
    (movies_df["budget"].isna()) |
    (movies_df["revenue"].isna()) |
    (movies_df["budget"] == 0) |
    (movies_df["revenue"] == 0)]
q2[["movie_id", "title", "budget", "revenue"]]

,movie_id,title,budget,revenue
324,791373,Zack Snyder's Justice League,70000000.0,0.0
340,405774,Bird Box,19800000.0,0.0
403,664413,365 Days,0.0,9458590.0
468,466282,To All the Boys I've Loved Before,0.0,0.0
554,766507,Prey,65000000.0,0.0
...,...,...,...,...
2469,755566,Day Shift,100000000.0,0.0
2471,73861,A Serbian Film,0.0,0.0
2480,11104,Chungking Express,160000.0,0.0
2481,500840,I'm Thinking of Ending Things,0.0,0.0


3
List the profit (revenue − budget) and return on investment for every movie that has valid budget and revenue figures.



In [17]:
q3 = movies_df[
    (movies_df["budget"] > 0) &
    (movies_df["revenue"] > 0)].copy()

q3["profit"] = q3["revenue"] - q3["budget"]
q3["roi"] = q3["profit"] / q3["budget"] * 100
q3[["title","revenue","budget", "profit", "roi"]]

,title,revenue,budget,profit,roi
0,Interstellar,7.466067e+08,1.650000e+08,5.816067e+08,352.488913
1,Inception,8.390306e+08,1.600000e+08,6.790306e+08,424.394144
2,The Avengers,1.518816e+09,2.200000e+08,1.298816e+09,590.370689
3,The Dark Knight,1.004558e+09,1.850000e+08,8.195584e+08,443.004564
4,Avatar,2.923706e+09,2.370000e+08,2.686706e+09,1133.631235
...,...,...,...,...,...
2498,Daybreakers,5.141719e+07,2.000000e+07,3.141719e+07,157.085940
2499,Date Night,8.759568e+07,9.321480e+07,-5.619119e+06,-6.028140
2500,Transformers,2.747467e+08,1.100988e+08,1.646479e+08,149.545660
2501,The SpongeBob Movie: Sponge Out of Water,5.706575e+08,3.928488e+07,5.313726e+08,1352.613653


4
List the top 15 movies by return on investment, considering only movies with a budget of at least $1 million.



In [18]:
q4 = movies_df[
    (movies_df["budget"] >= 1000000) & (movies_df["revenue"] > 0)].copy()

q4["roi"] = ((q4["revenue"] - q4["budget"]) / q4["budget"] * 100)

q4.sort_values("roi", ascending=False).head(15)[["movie_id", "title", "budget", "revenue", "roi"]]

,movie_id,title,budget,revenue,roi
1767,503314,Dragon Ball Super: Broly,1000000.0,125002821.0,12400.282100
535,408,Snow White and the Seven Dwarfs,1488423.0,184925486.0,12324.256142
1791,36685,The Rocky Horror Picture Show,1400000.0,171181400.0,12127.242857
467,1366,Rocky,1000000.0,117253345.0,11625.334500
1227,770,Gone with the Wind,4000000.0,402352579.0,9958.814475
716,9325,The Jungle Book,4000000.0,378000000.0,9350.000000
635,11224,Cinderella,2900000.0,263600000.0,8989.655172
372,176,Saw,1200000.0,104045735.0,8570.477917
719,12230,One Hundred and One Dalmatians,3600000.0,303000000.0,8316.666667
263,601,E.T. the Extra-Terrestrial,10500000.0,797307407.0,7493.403876


5
List the number of movies released per year, and identify which year had the highest movie output in the dataset.



In [19]:
movies_df["release_date"] = pd.to_datetime(movies_df["release_date"])

q5 = movies_df.groupby(movies_df["release_date"].dt.year).size()

print(q5)
print("Highest Year:", q5.idxmax())
print("Movies:", q5.max())

release_date
1921     1
1922     1
1927     1
1931     2
1936     1
        ..
2022    65
2023    62
2024    43
2025    26
2026     8
Length: 89, dtype: int64
Highest Year: 2016
Movies: 121


6
List all movies whose runtime is a statistical outlier compared to the rest of the dataset.



In [21]:
Q1 = movies_df["runtime"].quantile(.25)
Q3 = movies_df["runtime"].quantile(.75)
IQR = Q3 - Q1

q6 = movies_df[
    (movies_df["runtime"] < Q1 - 1.5*IQR) |
    (movies_df["runtime"] > Q3 + 1.5*IQR)]
q6[["title", "runtime"]]

,title,runtime
0,Interstellar,169
16,Avengers: Endgame,181
18,The Lord of the Rings: The Fellowship of the Ring,179
19,Titanic,194
20,The Lord of the Rings: The Return of the King,201
21,The Wolf of Wall Street,180
28,The Lord of the Rings: The Two Towers,179
38,The Godfather,175
76,The Hobbit: An Unexpected Journey,169
79,The Green Mile,189


7
List each movie's primary genre (the first genre listed) and
 compare average popularity across primary genres.



In [29]:
movie_genre_df=pd.read_csv("movie_genres_cleaned.csv")

In [34]:
df = movie_genre_df.merge(genres_df, on="genre_id").merge(
    movies_df, on="movie_id"
)

df = df.sort_values("genre_id")
q7 = df.groupby("movie_id").first()

q7.groupby("genre_name")["popularity"].mean().sort_values(ascending=False)

genre_name
Thriller           18.854481
Adventure          17.372862
Animation          15.234531
Horror             14.600246
Crime              14.592000
Western            13.714286
Action             13.284634
Science Fiction    12.597567
Fantasy            12.424829
Drama              11.151012
History            10.754000
Comedy             10.148475
Documentary         7.726200
War                 3.218400
Name: popularity, dtype: float64

8
List the top 10 most prolific actors by number of movies, along with their average movie rating across those movies.



In [37]:
df = cast_df.merge(movies_df, on="movie_id")

top10 = df.groupby("actor_name")["movie_id"].nunique().nlargest(10)

q8 = df.groupby("actor_name")["vote_average"].mean()

pd.DataFrame({"Movie Count": top10,"Average Rating": q8}).loc[top10.index]

,Movie Count,Average Rating
actor_name,,
Samuel L. Jackson,39.0,6.938692
Brad Pitt,38.0,7.257289
Robert De Niro,38.0,7.228316
Johnny Depp,37.0,6.856973
Tom Hanks,33.0,7.289758
Mark Wahlberg,32.0,6.630063
Scarlett Johansson,32.0,7.056344
Willem Dafoe,32.0,7.114781
Morgan Freeman,31.0,6.963742


9
List any movie titles that appear more than once in the dataset, along with their release years, to check for duplicates or remakes.



In [38]:
movies_df["release_year"] = pd.to_datetime(movies_df["release_date"]).dt.year

q9 = movies_df[movies_df["title"].duplicated(False)].sort_values("title")
q9[["title", "release_year"]]

,title,release_year
878,A Nightmare on Elm Street,1984
1928,A Nightmare on Elm Street,2010
261,Aladdin,1992
339,Aladdin,2019
166,Alice in Wonderland,2010
...,...,...
1737,The Thing,2011
744,Total Recall,1990
860,Total Recall,2012
252,Transformers,2007


10
List movies whose popularity score and vote count don't follow the general pattern between the two — i.e., unusually high popularity with a low vote count, or vice versa.



In [41]:
# Calculate Z-scores
movies_df["popularity_z"] = (
    movies_df["popularity"] - movies_df["popularity"].mean()
) / movies_df["popularity"].std()

movies_df["vote_count_z"] = (
    movies_df["vote_count"] - movies_df["vote_count"].mean()
) / movies_df["vote_count"].std()

# Difference between the two Z-scores
movies_df["z_diff"] = (movies_df["popularity_z"] - movies_df["vote_count_z"]).abs()

# Top 15 unusual movies
q10 = movies_df.sort_values(
    "z_diff", ascending=False).head(15)

q10[["movie_id",
    "title",
    "popularity",
    "vote_count",
    "z_diff"]]

,movie_id,title,popularity,vote_count,z_diff
2388,1275779,Disclosure Day,394.3346,2328,21.798752
46,634649,Spider-Man: No Way Home,438.4757,22429,20.182803
1315,1339713,Obsession,364.8722,4203,19.793731
2248,1083381,Backrooms,258.4743,2483,14.265255
713,687163,Project Hail Mary,162.7023,6750,8.116246
62,557,Spider-Man,180.5936,20991,6.232464
1465,936075,Michael,108.7248,3846,5.721196
2452,931285,Mortal Kombat II,99.5231,2260,5.532897
1707,1226863,The Super Mario Galaxy Movie,97.1210,3343,5.181862
35,315635,Spider-Man: Homecoming,169.0857,23456,5.099920
